## Exercise 14.3 (Engineered features and model performance)  

Repeat the comparison for HDF and PWF: train trees of increasing depth with and without the engineered features temp_diff and power. Report the F1 score instead of the accuracy and explain why accuracy is a misleading metric for this dataset.

After this exercise you should be able to:

1.  Choose an appropriate metric for heavily imbalanced classification tasks.

In [2]:
import numpy as np
import pandas as pd
import sklearn.tree as skt
from sklearn.metrics import f1_score


# Datensatz laden
df = pd.read_csv(
    "https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip"
)


# Berechnung der Temperaturdifferenz und Leistung
df["temp_diff"] = df["Process temperature [K]"] - df["Air temperature [K]"]

df["power"] = (
    df["Torque [Nm]"]
    * df["Rotational speed [rpm]"]
    * 2 * np.pi / 60
)


# Ausgabe der ersten Zeilen des DataFrames
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,temp_diff,power
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,10.5,6951.590560
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,10.5,6826.722724
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,10.4,7749.387543
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,10.4,5927.504659
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,10.5,5897.816608


# 1 Repeat the comparison for HDF and PWF: train trees of increasing depth with and without the engineered features temp_diff and power. Report the F1 score instead of the accuracy and explain why accuracy is a misleading metric for this dataset.

In [ ]:
from sklearn.metrics import f1_score

# Definition der Basismerkmale für die Klassifikation
base_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

# Definition der zusätzlichen Merkmale für die Klassifikation
extra_features = {
    "HDF": "temp_diff",     #Zeilvariable: HDF ; Zusatzmerkmal: temp_diff
    "PWF": "power"          #Zeilvariable: PWF ; Zusatzmerkmal: power
}

# Ergebnisse der einzelnen Modelle speichern
results = []    #Ergebnisliste initialisieren

for failure, extra_feature in extra_features.items():
    y = df[failure]     #Ergebnisvariable für den Baum festlegen HDF und PWF

    for depth in [1, 2, 4, 8]:                  #Tiefe der Bäume festlegen
        for use_extra in [False, True]:         #Flag für Verwendung des zusätzlichen Merkmals
            features = base_features.copy()

            #Wenn Zusatzmerkmal verwendet wird
            if use_extra:
                features.append(extra_feature)

            X = df[features]                    #Merkmale für den Baum festlegen

            #Einen Entscheidungsbaum erstellen
            tree = skt.DecisionTreeClassifier(
                max_depth=depth,                #maximale Tiefe des Baums
                random_state=42,
                min_samples_split=5,            #Mindestanzahl an Samples, die benötigt werden, um einen Knoten zu teilen
                min_samples_leaf=5              #Mindestanzahl an Samples, die benötigt werden, um ein Blatt zu sein
            )

            #Trainieren des Baums auf den Daten
            tree.fit(X, y)
            prediction = tree.predict(X)        #Vorhersage der Ergebnisse auf den Trainingsdaten

            #Ergebnisse in der Ergebnisliste speichern
            results.append({
                "Fehlerart": failure,
                "Tiefe": depth,
                "Zusatzmerkmal": extra_feature if use_extra else "ohne",
                "F1": f1_score(y, prediction, zero_division=0)             #Berechnung des F1-Scores für die Vorhersagen (Harmonisches Mittel von Präzision und Recall)
            })

# Ergebnisse in einem DataFrame speichern und anzeigen
results = pd.DataFrame(results)
results

,Fehlerart,Tiefe,Zusatzmerkmal,F1
0,HDF,1,ohne,0.000000
1,HDF,1,temp_diff,0.000000
2,HDF,2,ohne,0.000000
3,HDF,2,temp_diff,0.938776
4,HDF,4,ohne,0.543210
5,HDF,4,temp_diff,0.969432
6,HDF,8,ohne,0.929825
7,HDF,8,temp_diff,0.968610
8,PWF,1,ohne,0.553459
9,PWF,1,power,0.805031


Die berechneten Zusatzmerkmale verbessern die Erkennung der Ausfälle deutlich. Bei HDF steigt der F1-Score bei Tiefe 2 durch temp_diff von 0 auf etwa 0,939. Bei Tiefe 4 werden etwa 0,969 erreicht. Für PWF ermöglicht power bereits bei Tiefe 2 eine perfekte Trennung der Trainingsdaten mit F1 = 1. Ohne die Zusatzmerkmale benötigen die Bäume mehr Tiefe und erzielen trotzdem schlechtere Ergebnisse.

Accuracy wäre hier irreführend, weil Ausfälle sehr selten sind. Ein Modell, das immer „kein Ausfall“ vorhersagt, erreicht bereits über 98 % Accuracy. Der F1-Score berücksichtigt dagegen die Erkennung der Ausfälle und die Fehlalarme.

Die Ergebnisse beziehen sich auf die Trainingsdaten und zeigen, dass sinnvoll berechnete Merkmale die Modellierung vereinfachen. Die Vorhersagequalität für neue Daten wurde damit noch nicht geprüft.

HTML erstellt mit:

uv run jupyter nbconvert --to html "Tasks/14 Self study - Session 1/Exercise_14.3.ipynb"